# Real observed panels: high-level overview

You observe an outcome matrix **$O$** $\in \mathbb{R}^{N \times T}$ and a binary treatment mask **$Z$** $\in \{0,1\}^{N \times T}$ (units $\times$ time). [`causaltensor.real.estimate`](../../src/causaltensor/real/estimate.py) maps $(O, Z)$ to one or more point estimates of an average treated effect; the canonical string keys live in **`VALID_METHODS`**.

### What this notebook does

1. **Catalog built-in empirical panels** you can load with [`PanelDataset.from_builtin`](../../src/causaltensor/datasets/panel_dataset.py).
2. **Document the seven estimators** available through `estimate()`.
3. **Worked example**: load $\rightarrow$ inspect **$Z$** $\rightarrow$ **Step 2** point estimates $\rightarrow$ **Steps 3--4** explain the **table** plus **bootstrap bands** (caveats in Step 4 for single-treated designs).

Bring your own long or wide table when not using built-ins—the same $(O,Z)$ interface applies once you reshape to $(N,T)$.

> **Result objects:** for a full tour of what `fit()` returns — `summary()`, `plot_actual_vs_counterfactual()`, fixed effects, weights, diagnostics, etc. — see **[Guide 04 — Inspecting Results](./04_inspecting_results.ipynb)**.


## Built-in empirical panels

The keys below match `dataset_name` in **`PanelDataset.from_builtin(dataset_name)`** (backed by [`load_dataset`](../../src/causaltensor/datasets/dataset_loader.py)).

**Treatment patterns:** [`VALID_PATTERNS`](../../src/causaltensor/synthetic/dgp.py) lists **`IID`**, **`Block`**, **`Staggered`**, **`Adaptive`** (builders in [`treatment_patterns.py`](../../src/causaltensor/utils/treatment_patterns.py)). Built-in loaders use **`create_z_dataframe`**: one unit switches on at an event date and stays treated---an absorbing **Block** along that row (same shape as **`Z_block`** with **one** treated unit, i.e. $m_1=1$), not IID draws or **Adaptive** rules.

| Built-in key | Setting (treatment) | Outcome $Y$ | $Z$ | $X$? | Years | $N$ |
|:---:|:---|:---|:---:|:---:|:---:|:---:|
| `smoking` | California smoking ban, 1988 | `cigsale` | Block | Yes | 1970–2000 | 39 |
| `basque` | Basque region GDP shock, 1975 | `gdpcap` | Block | Yes | 1955–1997 | 18 |
| `german_reunification` | West Germany reunification, 1990 | `gdp` | Block | Yes | 1960–2003 | 17 |
| `texas` | Texas prison reform, 1993 | `bmprate` | Block | Yes | 1985–2000 | 51 |
| `pwt_spain_eu` | Spain joins EU, 1986 | `rgdpe` | Block | Yes | 1970–2000 | 156 |
| `pwt_chile_trade` | Chile trade liberalization, 1976 | `rgdpo` | Block | Yes | 1960–1995 | 111 |
| `pwt_korea_democracy` | Korean democratization, 1988 | `rgdpe` | Block | Yes | 1970–2000 | 156 |
| `pwt_norway_oil` | Norway Ekofisk oil, 1971 | `rgdpe` | Block | Yes | 1960–1980 | 111 |

For **Staggered**, **IID**, or **Adaptive** $Z$, build the panel yourself or use [`generate`](../../src/causaltensor/synthetic/dgp.py) / [`run_experiment`](../../src/causaltensor/semi_synthetic/experiment.py). **`available_datasets()`** mirrors the registry.


## Estimators

Each estimator is a `PanelSolver` subclass in [`causaltensor.cauest`](../../src/causaltensor/cauest/). Instantiate with `(O, Z)` and call `.fit()` to get back a result object with `.tau` (ATT scalar) and `.M` / `.baseline` (counterfactual panel, $N \times T$). All methods accept any binary $Z \in \{0,1\}^{N \times T}$. For MC-NNM, pass `cross_validation=True` (the default) to `fit()` to select the penalty via K-fold CV.

| Solver class | Intuition | Supports $X$? | `fit()` call |
|:---|:---|:---:|:---|
| [`DCPanelSolver`](../../src/causaltensor/cauest/DebiasConvex.py) | Convex low-rank + sparse debiasing (Athey et al. 2021). | No | `DCPanelSolver(O, Z).fit()` |
| [`MCNNMPanelSolver`](../../src/causaltensor/cauest/MCNNM.py) | Nuclear-norm matrix completion, CV-tuned penalty (Athey et al. 2021). | No | `MCNNMPanelSolver(O, Z).fit()` |
| [`CovariancePCAPanelSolver`](../../src/causaltensor/cauest/CovariancePCA.py) | Covariance PCA factor completion (Xiong & Pelger 2022). | No | `CovariancePCAPanelSolver(O, Z).fit()` |
| [`DIDPanelSolver`](../../src/causaltensor/cauest/DID.py) | Two-way FE DiD. | No | `DIDPanelSolver(O, Z).fit()` |
| [`SDIDPanelSolver`](../../src/causaltensor/cauest/SDID.py) | Synthetic DID with donor/time weights (Arkhangelsky et al. 2021). | Yes | `SDIDPanelSolver(O, Z).fit()` |
| [`OLSSCPanelSolver`](../../src/causaltensor/cauest/OLSSyntheticControl.py) | OLS synthetic control with simplex-constrained donor weights (Abadie et al. 2010). | Yes | `OLSSCPanelSolver(O, Z).fit()` |
| [`RSCPanelSolver`](../../src/causaltensor/cauest/RobustSyntheticControl.py) | Robust SC via low-rank donor SVD + linear extrapolation. | No | `RSCPanelSolver(O, Z).fit()` |

> **Counterfactual access:** use `res.M` for all solvers except MC-NNM (use `res.baseline`, which adds fixed effects to the low-rank `M`). All solvers return `M` in **(N, T)** shape.


In [7]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from causaltensor.datasets.panel_dataset import PanelDataset
from causaltensor.datasets.dataset_loader import available_datasets
from causaltensor.utils.common import treated_states_and_starts_from_Z

from causaltensor.cauest.DebiasConvex import DCPanelSolver
from causaltensor.cauest.MCNNM import MCNNMPanelSolver
from causaltensor.cauest.CovariancePCA import CovariancePCAPanelSolver
from causaltensor.cauest.DID import DIDPanelSolver
from causaltensor.cauest.SDID import SDIDPanelSolver
from causaltensor.cauest.OLSSyntheticControl import OLSSCPanelSolver
from causaltensor.cauest.RobustSyntheticControl import RSCPanelSolver

print("[Load] Built-in datasets:", available_datasets())

[Load] Built-in datasets: ('smoking', 'basque', 'german_reunification', 'texas', 'pwt_spain_eu', 'pwt_chile_trade', 'pwt_korea_democracy', 'pwt_norway_oil')


## Worked example: California smoking

Follow the numbered substeps below. **Inspect assignments before estimating** so contrasts match the empirical design.


In [8]:
# Change `dataset_name` to any key from the table above to try a different panel.
dataset_name = "smoking"
valid = available_datasets()
if dataset_name not in valid:
    raise ValueError(f"{dataset_name!r} not found. Available datasets: {valid}")

panel = PanelDataset.from_builtin(dataset_name)
O = panel.O.astype(float)
Z = panel.Z.astype(float)

print(f"[Load] {dataset_name!r}  shape {O.shape}  (units × periods)")
print(f"[Load] Units : {panel.unit_index[0]} … {panel.unit_index[-1]}")
print(f"[Load] Period: {int(panel.time_index.min())} → {int(panel.time_index.max())}")


[Load] 'smoking'  shape (39, 31)  (units × periods)
[Load] Units : Alabama … Wyoming
[Load] Period: 1970 → 2000


### Step 1 — Treatment design (`Z`) before estimation

Summarize **who** adopts and **when**. Single-unit sharp adoption (like CA) contrasts with richer staggered stacks you would encode manually elsewhere.


In [9]:
treated_rows, treat_start_cols = treated_states_and_starts_from_Z(Z)
design = pd.DataFrame({
    "unit": panel.unit_index[np.asarray(treated_rows, dtype=int)],
    "first_treat_period": panel.time_index[np.asarray(treat_start_cols, dtype=int)].astype(int),
})
share_treated_cells = float(Z.sum()) / Z.size
print("[Design] ever-treated rows:", len(treated_rows))
print("[Design] share of treated (i,t) cells:", round(share_treated_cells, 6))
display(design)

# Treatment mask heatmap (interactive)
Z_hot = pd.DataFrame(Z * 1.0, index=panel.unit_index, columns=panel.time_index.astype(int))
fig = px.imshow(
    Z_hot,
    color_continuous_scale=["white", "#2980b9"],
    zmin=0, zmax=1,
    labels={"x": "Period", "y": "Unit", "color": "Treated"},
    title=f"Treatment mask Z — {dataset_name!r}  (blue = treated)",
    aspect="auto",
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=max(300, Z.shape[0] * 18), margin=dict(l=120, r=20, t=50, b=40))
fig.show()


[Design] ever-treated rows: 1
[Design] share of treated (i,t) cells: 0.010753


,unit,first_treat_period
0,California,1988


### Step 2 — Estimate with every registered key


In [13]:
def _fit(name, O, Z):
    """Instantiate and run the appropriate solver class."""
    if name == "DC_PR":
        return DCPanelSolver(O, Z).fit()
    if name == "MC_NNM":
        return MCNNMPanelSolver(O, Z).fit()
    if name == "CovPCA":
        return CovariancePCAPanelSolver(O, Z).fit()
    if name == "DID":
        return DIDPanelSolver(O, Z).fit()
    if name == "SDID":
        return SDIDPanelSolver(O, Z).fit()
    if name == "SC":
        return OLSSCPanelSolver(O, Z).fit()
    if name == "RSC":
        return RSCPanelSolver(O, Z).fit()
    raise ValueError(f"Unknown solver: {name}")

SOLVERS = ("DC_PR", "MC_NNM", "CovPCA", "DID", "SDID", "SC", "RSC")
results, tau_hat = {}, {}
for name in SOLVERS:
    try:
        res = _fit(name, O, Z)
        results[name] = res
        tau_hat[name] = float(np.nanmean(np.asarray(res.tau).ravel()))
    except Exception as e:
        results[name] = None
        tau_hat[name] = np.nan
        print(f"[{name}] FAILED: {e}")

results_tbl = pd.DataFrame({"solver": list(SOLVERS), "tau_hat": [tau_hat[n] for n in SOLVERS]})
print("[Results] Point estimates (tau_hat):")
display(results_tbl)

bar_colors = ["#c0392b" if v < 0 else "#27ae60" for v in results_tbl["tau_hat"]]
fig = go.Figure(go.Bar(
    x=results_tbl["tau_hat"],
    y=results_tbl["solver"],
    orientation="h",
    marker_color=bar_colors,
))
fig.add_vline(x=0, line_width=1, line_color="black")
fig.update_layout(
    title=f"ATT estimates — {dataset_name!r}",
    xaxis_title="τ̂ (ATT)",
    yaxis_title="Solver",
    height=350,
    margin=dict(l=80, r=60, t=50, b=50),
)
fig.show()


[Results] Point estimates (tau_hat):


,solver,tau_hat
0,DC_PR,-13.721601
1,MC_NNM,-19.810581
2,CovPCA,-46.389542
3,DID,-26.485954
4,SDID,-15.393271
5,SC,-18.427808
6,RSC,-16.475112


### Step 3 — Visualize the outcome path

Plot the treated unit against the unweighted donor mean to inspect pre-trend alignment and the apparent post-treatment gap. A credible design shows parallel pre-treatment trends; a large divergence after the treatment date is the visual signature of a causal effect.

In [18]:
def _counterfactual(name, res, t_idx):
    """Extract the counterfactual time series for the first treated unit."""
    if res is None:
        return None
    if name == "MC_NNM":
        # res.baseline = low-rank M + fixed effects = true counterfactual
        return res.baseline[t_idx, :]
    if name == "SC":
        return res.M[t_idx, :]  # now (N, T) like all other solvers
    if name == "DID":
        # fitted_value = alpha_i + beta_t + tau*Z; subtract tau*Z to get counterfactual
        tau_scalar = float(np.asarray(res.tau).ravel()[0])
        return res.M[t_idx, :] - tau_scalar * Z[t_idx, :]
    return res.M[t_idx, :]   # DC_PR, SDID, CovPCA, RSC: M is the counterfactual directly

if treated_rows:
    t_idx        = treated_rows[0]
    t_start      = treat_start_cols[0]
    treat_year   = int(panel.time_index[t_start])
    treated_name = str(panel.unit_index[t_idx])
    times        = panel.time_index.astype(int)

    palette = px.colors.qualitative.Plotly
    fig = go.Figure()

    # Actual outcome (bold black)
    fig.add_trace(go.Scatter(
        x=times, y=O[t_idx, :],
        mode="lines",
        name="Actual",
        line=dict(color="black", width=3),
    ))

    # Counterfactual from each estimator
    all_cf_vals = []
    for i, name in enumerate(SOLVERS):
        cf = _counterfactual(name, results.get(name), t_idx)
        if cf is not None:
            all_cf_vals.append(cf)
            fig.add_trace(go.Scatter(
                x=times, y=cf,
                mode="lines",
                name=name,
                line=dict(color=palette[i % len(palette)], width=1.5, dash="dash"),
                opacity=0.9,
            ))

    # Treatment onset vertical line
    fig.add_vline(
        x=treat_year, line_dash="dot", line_color="grey", line_width=1.5,
        annotation_text=f"Treatment ({treat_year})",
        annotation_position="top left",
    )

    # Y-axis: cover actual data + all counterfactuals, clipped so outliers
    # don't collapse the scale. Use the Plotly toolbar to zoom out further.
    all_y = np.concatenate([O[t_idx, :]] + all_cf_vals)
    y_lo, y_hi = np.percentile(all_y, 2), np.percentile(all_y, 98)
    y_pad = 0.10 * (y_hi - y_lo)

    fig.update_layout(
        title=dict(
            text=f"Actual vs counterfactuals — {dataset_name!r} ({treated_name})",
            x=0.5, xanchor="center", y=0.97,
        ),
        xaxis_title="Period",
        yaxis_title="Outcome",
        yaxis=dict(range=[y_lo - y_pad, y_hi + y_pad]),
        height=500,
        legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
        margin=dict(l=60, r=20, t=60, b=120),
        hovermode="x unified",
    )
    fig.show()
else:
    print("[Step 3] No treated units detected in Z.")

### Step 4 — Inference

> **TODO:** Add per-estimator inference to the solvers.
>
> | Solver | Method | Status |
> |:---|:---|:---:|
> | `DCPanelSolver` | Sandwich SE → `res.std` (already computed) | ✅ Implemented |
> | `OLSSCPanelSolver` | Placebo-in-space permutation p-value (`pval=True`) — single treated unit only | ✅ Implemented |
> | `DIDPanelSolver` | OLS robust / cluster SE | ❌ Not yet |
> | `SDIDPanelSolver` | Jackknife variance (Arkhangelsky et al. 2021 § 5) | ❌ Not yet |
> | `MCNNMPanelSolver` | Bootstrap variance (Athey et al. 2021) | ❌ Not yet |
> | `CovariancePCAPanelSolver` | Bootstrap variance (Xiong & Pelger 2022) | ❌ Not yet |
> | `RSCPanelSolver` | Analytical SE / Bootstrap | ❌ Not yet |
>
> **Note on generalizability:** the SC placebo-in-space test is valid only for a single treated unit.
> For multiple treated units use estimator-specific methods (jackknife for SDID, sandwich SE for DC-PR)
> or a properly adapted Fisher randomization test that permutes the full $Z$ matrix.


## Optional preprocessing

Chain **`align` / `balance` / `impute` / `scale`** on [`PanelDataset`](../../src/causaltensor/datasets/panel_dataset.py); see **[`panel_dataset_quickstart.ipynb`](../panel_dataset_quickstart.ipynb)** for patterns. Bundled loaders already arrive on a rectangular grid.


## Next workflows

- **Synthetic Monte Carlos** with known $\tau$: [`causaltensor.synthetic.generate`](../../src/causaltensor/synthetic/).
- **Semi-synthetic stress tests**: [`causaltensor.semi_synthetic.run_experiment`](../../src/causaltensor/semi_synthetic/experiment.py).

Companion guides for those tracks belong in **`tutorials/guides/`**.
